# 03 — Statistical Analysis
Paired t-tests, Wilcoxon, effect sizes

In [1]:
import numpy as np
from scipy import stats
import importlib.util
spec = importlib.util.spec_from_file_location("cfg", "00_config.py")
cfg = importlib.util.module_from_spec(spec); spec.loader.exec_module(cfg)
for a in dir(cfg):
    if not a.startswith('_'): globals()[a] = getattr(cfg, a)

all_subjs = np.load(CKPT['subjects'], allow_pickle=True).tolist()
erp_results = np.load(CKPT['erp_results'], allow_pickle=True)['erp_results'].tolist()

tgt = {c:[] for c in ERP_COMPONENTS}
ntgt = {c:[] for c in ERP_COMPONENTS}
for r in erp_results:
    for c in ERP_COMPONENTS:
        (tgt if r['condition']=='Target' else ntgt)[c].append(r[f'{c}_mean'])

In [2]:
# paired t-tests + effect sizes
print("PAIRED T-TESTS")
print("-" * 55)
for comp in ERP_COMPONENTS:
    t_arr, n_arr = np.array(tgt[comp]), np.array(ntgt[comp])
    t_stat, p = stats.ttest_rel(t_arr, n_arr)
    d = np.mean(t_arr - n_arr) / np.std(t_arr - n_arr, ddof=1)
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    print(f"  {comp}: tgt={np.mean(t_arr):.2f} ntgt={np.mean(n_arr):.2f} t={t_stat:.2f} p={p:.4f}{sig} d={d:.2f}")

PAIRED T-TESTS
-------------------------------------------------------
  N170: tgt=-2.13 ntgt=0.59 t=-4.13 p=0.0026** d=-1.31
  VPP: tgt=1.47 ntgt=-0.19 t=2.72 p=0.0234* d=0.86
  P300: tgt=4.30 ntgt=-0.07 t=6.38 p=0.0001*** d=2.02


In [3]:
# wilcoxon (better for N=10)
print("\nWILCOXON SIGNED-RANK")
print("-" * 45)
for comp in ERP_COMPONENTS:
    t_arr, n_arr = np.array(tgt[comp]), np.array(ntgt[comp])
    try:
        w, p = stats.wilcoxon(t_arr, n_arr)
        sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
        print(f"  {comp}: W={w:.0f} p={p:.4f} {sig}")
    except Exception as e:
        print(f"  {comp}: failed ({e})")


WILCOXON SIGNED-RANK
---------------------------------------------
  N170: W=3 p=0.0098 **
  VPP: W=6 p=0.0273 *
  P300: W=0 p=0.0020 **


In [4]:
# effect size summary
print("\nEFFECT SIZES")
print("-" * 45)
print(f"{'Comp':<6} {'d':>6} {'Size':<10} {'p':>8}")
for comp in ERP_COMPONENTS:
    t_arr, n_arr = np.array(tgt[comp]), np.array(ntgt[comp])
    diff = t_arr - n_arr
    _, p = stats.ttest_rel(t_arr, n_arr)
    d = np.mean(diff) / np.std(diff, ddof=1)
    sz = 'large' if abs(d)>=0.8 else 'medium' if abs(d)>=0.5 else 'small'
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else ''
    print(f"  {comp:<6} {d:>+6.2f} {sz:<10} {p:>7.4f}{sig}")


EFFECT SIZES
---------------------------------------------
Comp        d Size              p
  N170    -1.31 large       0.0026**
  VPP     +0.86 large       0.0234*
  P300    +2.02 large       0.0001***
